# The small run, stage by stage

One shot of the small example (d=3, 12 rounds, 3 sliding windows) walked
through **every stage of the pipeline**. Every stage shows the **actual
data**: the object that goes in, the object that comes out, and the
timestamps.

```
QPU -> QC link -> controller (pulses to binary) -> syndrome packing
    -> C2B link -> Buffer 0 -> window manager -> CWD link -> decoder memory
    -> decoder engine (fetch -> algorithm -> release) -> DD handoff
    -> WDO link -> Pauli frame commit
```

The run uses p = 0.001 and seed 8, chosen so a real defect flows through:
one physical error between rounds 10 and 11 fires one detector bit, the
logical observable really flips, and window 2's decode catches it. Stage
costs are preset cards, so every timestamp is still exactly the hand
arithmetic of README section 1; the last cell checks that, cell for cell.

One honesty note: the run **releases** its transient messages after each
stage consumes them (Buffer 0 frees a round once its windows have read it).
Where a cell shows such an object, it rebuilds it with the run's own
message classes from the same sampled bits, so it is field for field what
flowed.

In [1]:
# Make the repository root and this folder importable, wherever jupyter started.
import sys
from pathlib import Path

repo_root = Path.cwd().resolve()
while not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "guide" / "walkthrough"))
sys.path.insert(0, str(repo_root))
print(repo_root)

/scratch/gpfs/MARTONOSI/sk2415/qlx-qec-sandbox/decsim


## The configuration: one yaml names every cost

Round period 1.0 µs. QC 0.15, C2B 0.10, CWD 2.0, DD 0.5, WDO 1.0 (all
propagation only, unbounded bandwidth). Controller processing and packing
0. Engine 250 MHz (0.004 µs per cycle), algorithm card 0.028, frame commit
0.004.

In [2]:
from experiments.baseline.baseline_closed_loop import build_run, load_config

CONFIG_PATH = repo_root / "experiments/validation/analytic_oracle_d3_r12.yaml"
print(CONFIG_PATH.read_text())

# Deterministic analytical-oracle case.
#
# Purpose:
#   Verify the exact baseline event ordering and timing, not performance or LER.
#   This case intentionally has:
#     - zero physical noise,
#     - one fixed decoder latency,
#     - one decoder unit,
#     - unbounded link bandwidth,
#     - only 12 QEC rounds, which produce exactly 3 sliding windows.
#
# Expected analytical results are provided beside this file.

code_task: surface_code:rotated_memory_z
distance: 3
rounds_per_shot: 12

windowing:
  scheme: sliding
  commit_rounds: 3
  buffer_rounds: 3

sweep:
  # p = 0 exactly cannot run: a noiseless circuit has an empty detector error
  # model and no matching graph. 1.0e-9 samples zero defects on every shot and
  # every timing tick is identical to p = 0 (preset stage costs, payload sizes
  # independent of p).
  - physical_error_probability: [1.0e-9]
    round_period_us: [1.0]
    algorithm_latency_us: [0.028]
    shots: 1

controller:
  t_binary_availability_us: 0.0
  t_pack

## Run the shot

`build_run` assembles the path from the yaml; `spec.build()` replays it in
the event loop. Every later cell reads the records this run left behind:
link transfers, window timestamps, engine stage records, Pauli frame
records, and the device's sampled bits.

In [3]:
from decsim.config import TICKS_PER_US

config = load_config(CONFIG_PATH)
spec, decoder_engine = build_run(config,
                                 physical_error_probability=0.001,
                                 round_period_us=1.0,
                                 algorithm_latency_us=0.028,
                                 seed=8)
completed = spec.build()

transfers = completed.result.link_traffic["transfers"]
windows = {window_id: window
           for (_, window_id), window in sorted(completed.window_manager.windows.items())}
frame_records = {record.window_key[1]: record
                 for record in completed.pauli_frame.snapshot().records}
operation = spec.ops[0]
model = completed.qpu.model


def us(ticks):
    return ticks / TICKS_PER_US


def link(path):
    """round or window id -> this path's transfer record."""
    key = "round_lo" if path in ("qc", "c2b") else "window_id"
    return {t["attribution"][key]: t for t in transfers if t["path"] == path}


qc, c2b = link("qc"), link("c2b")
cwd, dd, wdo = link("cwd"), link("dd"), link("wdo")


def readout(round_index):
    """The QPU's actual emitted fragment for this round."""
    return model.round_payloads(operation, round_index)[0]


def bit_string(bits):
    return "".join(str(int(bit)) for bit in bits)


round_bits = {round_index: bit_string(readout(round_index).bits)
              for round_index in sorted(qc)}
print(completed.result.terminal_status)

complete


## Stage 1: the QPU

**IN (from the controller): nothing at runtime.** The controller-to-QPU
link (`cq`) is unused in this baseline; no instruction traffic flows during
the shot. The QPU received its whole program when the run was built: the
`Operation` below, whose circuit it replays one round per microsecond.

In [4]:
print("paths that carried traffic:", sorted({t["path"] for t in transfers}))
print()
print("IN, installed at build time:")
print(f"Operation(id={operation.id}, name={operation.name!r}, qubits={operation.qubits}, "
      f"patches={operation.patches},")
print(f"          circuit=<{len(operation.circuit)} instructions, "
      f"{operation.circuit.num_detectors} detectors, "
      f"{operation.circuit.num_observables} observable>)")

paths that carried traffic: ['c2b', 'cwd', 'dd', 'qc', 'wdo']

IN, installed at build time:
Operation(id=1, name='memory', qubits=(0,), patches=(0,),
          circuit=<56 instructions, 96 detectors, 1 observable>)


In [5]:
print(operation.circuit)

QUBIT_COORDS(1, 1) 1
QUBIT_COORDS(2, 0) 2
QUBIT_COORDS(3, 1) 3
QUBIT_COORDS(5, 1) 5
QUBIT_COORDS(1, 3) 8
QUBIT_COORDS(2, 2) 9
QUBIT_COORDS(3, 3) 10
QUBIT_COORDS(4, 2) 11
QUBIT_COORDS(5, 3) 12
QUBIT_COORDS(6, 2) 13
QUBIT_COORDS(0, 4) 14
QUBIT_COORDS(1, 5) 15
QUBIT_COORDS(2, 4) 16
QUBIT_COORDS(3, 5) 17
QUBIT_COORDS(4, 4) 18
QUBIT_COORDS(5, 5) 19
QUBIT_COORDS(4, 6) 25
R 1 3 5 8 10 12 15 17 19
X_ERROR(0.001) 1 3 5 8 10 12 15 17 19
R 2 9 11 13 14 16 18 25
X_ERROR(0.001) 2 9 11 13 14 16 18 25
TICK
DEPOLARIZE1(0.001) 1 3 5 8 10 12 15 17 19
H 2 11 16 25
DEPOLARIZE1(0.001) 2 11 16 25
TICK
CX 2 3 16 17 11 12 15 14 10 9 19 18
DEPOLARIZE2(0.001) 2 3 16 17 11 12 15 14 10 9 19 18
TICK
CX 2 1 16 15 11 10 8 14 3 9 12 18
DEPOLARIZE2(0.001) 2 1 16 15 11 10 8 14 3 9 12 18
TICK
CX 16 10 11 5 25 19 8 9 17 18 12 13
DEPOLARIZE2(0.001) 16 10 11 5 25 19 8 9 17 18 12 13
TICK
CX 16 8 11 3 25 17 1 9 10 18 5 13
DEPOLARIZE2(0.001) 16 8 11 3 25 17 1 9 10 18 5 13
TICK
H 2 11 16 25
DEPOLARIZE1(0.001) 2 11 16 25
TICK
X

**OUT: one `QPUReadout` fragment per round**, the round's detector bits.
Round r finishes, and its fragment leaves the QPU, at r x 1.000 µs. Watch
round 11: one bit is set, the defect.

In [6]:
for round_index in (1, 2, 11, 12):
    fragment = readout(round_index)
    print(f"t={us(qc[round_index]['send_ticks']):6.3f} µs  OUT: "
          f"QPUReadout(operation_id={fragment.operation_id}, patch_id={fragment.patch_id}, "
          f"round_index={fragment.round_index}, bits={bit_string(fragment.bits)}, "
          f"n_fragments={fragment.n_fragments}, fragment_index={fragment.fragment_index}, "
          f"size_bits={fragment.size_bits})")
print()
print("round | leaves QPU (µs) | detector bits | fired")
for round_index, bits in round_bits.items():
    print(f"{round_index:5} | {us(qc[round_index]['send_ticks']):15.3f} "
          f"| {bits:>13} | {bits.count('1')}")

t= 1.000 µs  OUT: QPUReadout(operation_id=1, patch_id=0, round_index=1, bits=0000, n_fragments=1, fragment_index=0, size_bits=4)
t= 2.000 µs  OUT: QPUReadout(operation_id=1, patch_id=0, round_index=2, bits=00000000, n_fragments=1, fragment_index=0, size_bits=8)
t=11.000 µs  OUT: QPUReadout(operation_id=1, patch_id=0, round_index=11, bits=00010000, n_fragments=1, fragment_index=0, size_bits=8)
t=12.000 µs  OUT: QPUReadout(operation_id=1, patch_id=0, round_index=12, bits=000000000000, n_fragments=1, fragment_index=0, size_bits=12)

round | leaves QPU (µs) | detector bits | fired
    1 |           1.000 |          0000 | 0
    2 |           2.000 |      00000000 | 0
    3 |           3.000 |      00000000 | 0
    4 |           4.000 |      00000000 | 0
    5 |           5.000 |      00000000 | 0
    6 |           6.000 |      00000000 | 0
    7 |           7.000 |      00000000 | 0
    8 |           8.000 |      00000000 | 0
    9 |           9.000 |      00000000 | 0
   10 |          10.

## Stage 2: the QC link (QPU -> controller)

**IN**: the `QPUReadout` fragment at its send tick.
**OUT**: the same bits accepted at the controller 0.150 µs later as a
`SyndromePayload`, the class for "one binary detector-data round accepted
by the controller".

In [7]:
from decsim.message import SyndromePayload, normalize_binary_bits

fragment = readout(11)
accepted = SyndromePayload(operation_id=fragment.operation_id, patch_id=fragment.patch_id,
                           round_index=fragment.round_index,
                           bits=normalize_binary_bits(fragment.bits), code=fragment.code,
                           n_fragments=fragment.n_fragments,
                           fragment_index=fragment.fragment_index,
                           size_bits=fragment.size_bits)
print(f"t={us(qc[11]['delivery_ticks']):.3f} µs  OUT (round 11): {accepted}")
print()
print("round | bits | sent (µs) | at controller (µs) | delay (µs)")
for round_index in sorted(qc):
    transfer = qc[round_index]
    print(f"{round_index:5} | {transfer['payload_bits']:4} "
          f"| {us(transfer['send_ticks']):9.3f} | {us(transfer['delivery_ticks']):18.3f} "
          f"| {us(transfer['delivery_ticks'] - transfer['send_ticks']):10.3f}")

t=11.150 µs  OUT (round 11): SyndromePayload(operation_id=1, patch_id=0, round_index=11, bits=(0, 0, 0, 1, 0, 0, 0, 0), code=None, n_fragments=1, fragment_index=0, size_bits=8)

round | bits | sent (µs) | at controller (µs) | delay (µs)
    1 |    4 |     1.000 |              1.150 |      0.150
    2 |    8 |     2.000 |              2.150 |      0.150
    3 |    8 |     3.000 |              3.150 |      0.150
    4 |    8 |     4.000 |              4.150 |      0.150
    5 |    8 |     5.000 |              5.150 |      0.150
    6 |    8 |     6.000 |              6.150 |      0.150
    7 |    8 |     7.000 |              7.150 |      0.150
    8 |    8 |     8.000 |              8.150 |      0.150
    9 |    8 |     9.000 |              9.150 |      0.150
   10 |    8 |    10.000 |             10.150 |      0.150
   11 |    8 |    11.000 |             11.150 |      0.150
   12 |   12 |    12.000 |             12.150 |      0.150


## Stage 3: controller processing (pulses to binary)

**IN**: the delivered payload.
**OUT**: the same bits, declared binary-available after
`t_binary_availability_us`. This yaml sets it to 0.0 (readout
classification is priced inside the round), so availability = delivery and
the data is unchanged.

In [8]:
t_binary_us = config["controller"]["t_binary_availability_us"]
print(f"t_binary_availability_us = {t_binary_us}")
print("round | at controller (µs) | binary available (µs)")
for round_index in sorted(qc):
    delivered_us = us(qc[round_index]["delivery_ticks"])
    print(f"{round_index:5} | {delivered_us:18.3f} | {delivered_us + t_binary_us:21.3f}")

t_binary_availability_us = 0.0
round | at controller (µs) | binary available (µs)
    1 |              1.150 |                 1.150
    2 |              2.150 |                 2.150
    3 |              3.150 |                 3.150
    4 |              4.150 |                 4.150
    5 |              5.150 |                 5.150
    6 |              6.150 |                 6.150
    7 |              7.150 |                 7.150
    8 |              8.150 |                 8.150
    9 |              9.150 |                 9.150
   10 |             10.150 |                10.150
   11 |             11.150 |                11.150
   12 |             12.150 |                12.150


## Stage 4: syndrome packing

**IN**: every retained fragment of round r. This device emits one fragment
per round, so the input is a tuple of one `RetainedSyndromeFragment` and
there is never a wait for missing fragments.
**OUT**: one `SyndromeRoundPacket`, the round complete and immutable,
ready after `t_pack_us` (0.0, packing one fragment is a no-op). Note the
bits are the same 8 bits that left the QPU; packing changes the container,
not the data.

In [9]:
from decsim.message import RetainedSyndromeFragment, SyndromeRoundPacket

t_pack_us = config["controller"]["t_pack_us"]


def retained(round_index):
    fragment = readout(round_index)
    return RetainedSyndromeFragment(
        operation_id=fragment.operation_id, patch_id=fragment.patch_id,
        round_index=fragment.round_index,
        bits=tuple(int(bit) for bit in fragment.bits), code=fragment.code,
        size_bits=fragment.size_bits, fragment_index=fragment.fragment_index)


packed_us = us(qc[11]["delivery_ticks"]) + t_binary_us + t_pack_us
print(f"IN  (round 11): ({retained(11)},)")
packet = SyndromeRoundPacket(operation_id=operation.id, round_index=11,
                             fragments=(retained(11),))
print(f"\nt={packed_us:.3f} µs  OUT (round 11): {packet}")
print()
print("round | fragments in | bits in | packet bits out | packed at (µs)")
for round_index in sorted(qc):
    fragment = readout(round_index)
    at_us = us(qc[round_index]["delivery_ticks"]) + t_binary_us + t_pack_us
    print(f"{round_index:5} | {fragment.n_fragments:12} | {fragment.size_bits:7} "
          f"| {c2b[round_index]['payload_bits']:15} | {at_us:14.3f}")

IN  (round 11): (RetainedSyndromeFragment(operation_id=1, patch_id=0, round_index=11, bits=(0, 0, 0, 1, 0, 0, 0, 0), code=None, size_bits=8, fragment_index=0),)

t=11.150 µs  OUT (round 11): SyndromeRoundPacket(operation_id=1, round_index=11, fragments=(RetainedSyndromeFragment(operation_id=1, patch_id=0, round_index=11, bits=(0, 0, 0, 1, 0, 0, 0, 0), code=None, size_bits=8, fragment_index=0),))

round | fragments in | bits in | packet bits out | packed at (µs)
    1 |            1 |       4 |               4 |          1.150
    2 |            1 |       8 |               8 |          2.150
    3 |            1 |       8 |               8 |          3.150
    4 |            1 |       8 |               8 |          4.150
    5 |            1 |       8 |               8 |          5.150
    6 |            1 |       8 |               8 |          6.150
    7 |            1 |       8 |               8 |          7.150
    8 |            1 |       8 |               8 |          8.150
    9 

## Stage 5: the C2B link (controller -> Buffer 0)

**IN**: the `SyndromeRoundPacket` at its send tick.
**OUT**: the same packet **published** in Buffer 0, 0.100 µs later.
Publication is the moment the window manager can see the round; Buffer 0
retains the packet until every window that reads the round has consumed
it, then releases it (which is why the notebook rebuilds packets above
instead of reading them out of the finished buffer). Rounds land every
1.000 µs, a constant 0.250 µs after they left the QPU: links shift the
phase, never the rate.

In [10]:
print("round | sent (µs) | published in Buffer 0 (µs) | rounds published so far")
for round_index in sorted(c2b):
    transfer = c2b[round_index]
    print(f"{round_index:5} | {us(transfer['send_ticks']):9.3f} "
          f"| {us(transfer['delivery_ticks']):26.3f} | {round_index:22}")

round | sent (µs) | published in Buffer 0 (µs) | rounds published so far
    1 |     1.150 |                      1.250 |                      1
    2 |     2.150 |                      2.250 |                      2
    3 |     3.150 |                      3.250 |                      3
    4 |     4.150 |                      4.250 |                      4
    5 |     5.150 |                      5.250 |                      5
    6 |     6.150 |                      6.250 |                      6
    7 |     7.150 |                      7.250 |                      7
    8 |     8.150 |                      8.250 |                      8
    9 |     9.150 |                      9.250 |                      9
   10 |    10.150 |                     10.250 |                     10
   11 |    11.150 |                     11.250 |                     11
   12 |    12.150 |                     12.250 |                     12


## Stage 6: the window manager (rounds -> decode jobs)

**IN**: the published rounds.
**OUT**: one `DecodeJob` per window: the window's identity, its rounds, and
its data, which is simply those rounds' bits side by side. A window is
**ready** when the last round it reads is published, **queued** once its
dependency (the previous window's DD handoff) has arrived, **dispatched**
when a decoder unit is free. Window 2's 52 bits contain the fired bit from
round 11.

In [11]:
last_round = config["rounds_per_shot"]
window_reads = {window_id: (window.start_round, min(window.buffer_hi, last_round))
                for window_id, window in windows.items()}
window_bits = {}
for window_id, (read_lo, read_hi) in window_reads.items():
    window_bits[window_id] = "".join(round_bits[r] for r in range(read_lo, read_hi + 1))

for window_id, window in windows.items():
    read_lo, read_hi = window_reads[window_id]
    print(f"OUT: DecodeJob(op_id={window.op_id}, window_id={window_id}, "
          f"n_rounds={window.n_rounds}, rounds={read_lo}..{read_hi},")
    print(f"               bits={window_bits[window_id]})")
print()
print("window | ready (µs) | depends on | queued (µs) | dispatch (µs)")
for window_id, window in windows.items():
    dependencies = ",".join(str(dep[1]) for dep in window.deps) or "-"
    print(f"{window_id:6} | {us(window.t_data_complete):10.3f} | {dependencies:>10} "
          f"| {us(window.t_queued):11.3f} | {us(window.t_dispatch):12.3f}")

OUT: DecodeJob(op_id=1, window_id=0, n_rounds=6, rounds=1..6,
               bits=00000000000000000000000000000000000000000000)
OUT: DecodeJob(op_id=1, window_id=1, n_rounds=6, rounds=4..9,
               bits=000000000000000000000000000000000000000000000000)
OUT: DecodeJob(op_id=1, window_id=2, n_rounds=6, rounds=7..12,
               bits=0000000000000000000000000000000000010000000000000000)

window | ready (µs) | depends on | queued (µs) | dispatch (µs)
     0 |      6.250 |          - |       6.250 |        6.250
     1 |      9.250 |          0 |       9.250 |        9.250
     2 |     12.250 |          1 |      12.250 |       12.250


## Stage 7: the CWD link into decoder memory

**IN**: the dispatched job's rounds out of Buffer 0 (44, 48, 52 bits: the
actual window data).
**OUT**: the same bits sitting in the decoder unit's input memory 2.000 µs
later. Link delivery IS memory arrival here: the memory is unbounded
(`decoder_memory_rounds: null`), so admission never blocks and adds no
separate timestamp.

In [12]:
print("window | bits transferred | leaves Buffer 0 (µs) | in decoder memory (µs)")
for window_id in windows:
    transfer = cwd[window_id]
    print(f"{window_id:6} | {transfer['payload_bits']:16} "
          f"| {us(transfer['send_ticks']):20.3f} | {us(transfer['delivery_ticks']):22.3f}")

window | bits transferred | leaves Buffer 0 (µs) | in decoder memory (µs)
     0 |               44 |                6.250 |                  8.250
     1 |               48 |                9.250 |                 11.250
     2 |               52 |               12.250 |                 14.250


## Stage 8: the decoder engine (fetch -> algorithm -> release)

**IN**: the window's bits, read back out of decoder memory by fetch
(6 rounds x 0.004 µs), plus the window's detector error model slice.
**OUT**: the window's **correction**: its logical observable estimate.
The algorithm card charges 0.028 µs, release writes the correction out in
one cycle. Windows 0 and 1 saw all-zero syndromes and output 0; window 2
saw the fired bit and outputs 1.

In [13]:
print("window | bits in -> correction out | fetch (µs) | algorithm (µs) | release (µs) | done (µs)")
for window_id, window in windows.items():
    stages = {record.stage: record
              for record in decoder_engine.stage_records_for(operation.id, window_id)}
    def span(name):
        record = stages[name]
        return f"{us(record.start_ticks):.3f}-{us(record.end_ticks):.3f}"
    correction = frame_records[window_id].logical_observables
    fired = window_bits[window_id].count("1")
    print(f"{window_id:6} | {len(window_bits[window_id]):3} bits ({fired} fired) -> {correction} "
          f"| {span('fetch'):>13} | {span('algorithm'):>14} | {span('release'):>13} "
          f"| {us(window.t_done):9.3f}")

window | bits in -> correction out | fetch (µs) | algorithm (µs) | release (µs) | done (µs)
     0 |  44 bits (0 fired) -> (0,) |   8.250-8.274 |    8.274-8.302 |   8.302-8.306 |     8.306
     1 |  48 bits (0 fired) -> (0,) | 11.250-11.274 |  11.274-11.302 | 11.302-11.306 |    11.306
     2 |  52 bits (1 fired) -> (1,) | 14.250-14.274 |  14.274-14.302 | 14.302-14.306 |    14.306


## Stage 9: the DD handoff (decoder -> next window's decoder)

**IN**: the decoded window's boundary information, at decode done.
**OUT**: delivered to the next window's decode 0.500 µs later; that
delivery is exactly the dependency arrival that gated stage 6. On the wire
it is priced as the reference card's representative 100-bit boundary
transaction; in sliding windows the semantic boundary rides in the overlap
rounds the next window already reads (window 1 reads 4..9, overlapping
window 0's 4..6). The last window has no successor and sends nothing.

In [14]:
print("window | payload bits | sent (µs) | delivered to next window (µs)")
for window_id in windows:
    if window_id in dd:
        transfer = dd[window_id]
        print(f"{window_id:6} | {transfer['payload_bits']:12} | {us(transfer['send_ticks']):9.3f} "
              f"| {us(transfer['delivery_ticks']):29.3f}")
    else:
        print(f"{window_id:6} | (last window, no handoff)")

window | payload bits | sent (µs) | delivered to next window (µs)
     0 |          100 |     8.306 |                         8.806
     1 |          100 |    11.306 |                        11.806
     2 | (last window, no handoff)


## Stage 10: the WDO link (decoder -> Pauli frame)

**IN**: the correction message, `(window_key, logical_observables)`,
leaving the decoder at decode done.
**OUT**: the same message at the Pauli frame 1.000 µs later. The wire size
is the reference card's representative weak-output payload; the semantic
content is one observable bit per window.

In [15]:
print("window | message | leaves decoder (µs) | at Pauli frame (µs)")
for window_id, record in sorted(frame_records.items()):
    transfer = wdo[window_id]
    message = f"(window_key={record.window_key}, observables={record.logical_observables})"
    print(f"{window_id:6} | {message:44} | {us(transfer['send_ticks']):19.3f} "
          f"| {us(transfer['delivery_ticks']):19.3f}")

window | message | leaves decoder (µs) | at Pauli frame (µs)
     0 | (window_key=(1, 0), observables=(0,))        |               8.306 |               9.306
     1 | (window_key=(1, 1), observables=(0,))        |              11.306 |              12.306
     2 | (window_key=(1, 2), observables=(1,))        |              14.306 |              15.306


## Stage 11: the Pauli frame commit

**IN**: the delivered corrections, in window order.
**OUT**: the running frame: the XOR of everything committed so far,
finalized 0.004 µs after each acceptance. The final frame is the loop's
logical prediction. The truth is the QPU's own sampled observable flip:
they agree, so the shot is decoded correctly even though the observable
really flipped.

In [16]:
frame_value = 0
print("window | accepted (µs) | committed (µs) | correction | frame after commit")
for window_id, record in sorted(frame_records.items()):
    frame_value ^= record.logical_observables[0]
    print(f"{window_id:6} | {us(record.accepted_ticks):13.3f} "
          f"| {us(record.committed_ticks):14.3f} | {record.logical_observables} | ({frame_value},)")

operation_result = completed.result.operation_results[0]
print(f"\nloop prediction: {operation_result.logical_observables}")
print(f"observable truth: {operation_result.observable_truth}")
print(f"logical failure: {operation_result.logical_observables != tuple(operation_result.observable_truth)}")

window | accepted (µs) | committed (µs) | correction | frame after commit
     0 |         9.306 |          9.310 | (0,) | (0,)
     1 |        12.306 |         12.310 | (0,) | (0,)
     2 |        15.306 |         15.310 | (1,) | (1,)

loop prediction: (1,)
observable truth: (1,)
logical failure: False


## The end-to-end table, checked against the hand arithmetic

The same run condensed to one row per window, compared cell for cell
against README section 1's closed-form answer key. The defect changed the
*data* (round 11's bit, window 2's 52-bit input, window 2's correction,
the final frame), but not one *timestamp*: stage costs are preset cards,
so timing is independent of the noise.

In [17]:
from analytic_small_run import COLUMNS, analytic_timeline, print_table
from simulated_small_run import differences

simulated_rows = []
for window_id, window in windows.items():
    read_lo, read_hi = window_reads[window_id]
    record = frame_records[window_id]
    simulated_rows.append({
        "window_id": window_id, "read_lo": read_lo, "read_hi": read_hi,
        "commit_lo": window.commit_lo, "commit_hi": window.commit_hi,
        "buffer0_ready_us": us(window.t_data_complete),
        "queued_us": us(window.t_queued),
        "dispatch_us": us(window.t_dispatch),
        "decode_done_us": us(window.t_done),
        "dd_delivery_us": us(dd[window_id]["delivery_ticks"]) if window_id in dd else None,
        "frame_commit_us": us(record.committed_ticks),
        "buffer0_ready_to_frame_us": us(record.committed_ticks - window.t_data_complete),
    })
print_table(simulated_rows)

disagreements = differences(analytic_timeline(config), simulated_rows)
assert not disagreements, disagreements
print(f"\nMATCH: all {len(simulated_rows) * len(COLUMNS)} cells agree to the tick.")

| window_id | read_lo | read_hi | commit_lo | commit_hi | buffer0_ready_us | queued_us | dispatch_us | decode_done_us | dd_delivery_us | frame_commit_us | buffer0_ready_to_frame_us |
|---|---|---|---|---|---|---|---|---|---|---|---|
| 0 | 1 | 6 | 1 | 3 | 6.250 | 6.250 | 6.250 | 8.306 | 8.806 | 9.310 | 3.060 |
| 1 | 4 | 9 | 4 | 6 | 9.250 | 9.250 | 9.250 | 11.306 | 11.806 | 12.310 | 3.060 |
| 2 | 7 | 12 | 7 | 12 | 12.250 | 12.250 | 12.250 | 14.306 |  | 15.310 | 3.060 |

MATCH: all 36 cells agree to the tick.


## Try a knob

Change one number in the yaml (`cwd.latency_us`, `round_period_us`, the
seed above) and rerun all cells: every stage table shifts exactly as the
five formulas of README section 1 predict, and the final check must still
say MATCH (unless you changed a cost, in which case redo the arithmetic
first and watch it agree again). The frequency sweep and figures are README
section 5.